# 09 LUNAR Epochs Analysis v2

In [ ]:

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from metrics import find_best_f1_threshold, evaluate_scores, minmax_scale_scores
from results import build_experiment_record, save_record_json, get_memory_mb
from ensemble_utils import (
    tune_if, tune_lof, tune_dbscan, tune_ocsvm,
    score_if, score_lof, score_dbscan, score_ocsvm,
    tune_meta_fusion, apply_meta_fusion,
)

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200
META_TRIALS = 60
FUSION_STRATEGIES = ["mean", "max", "weighted", "rank_mean", "stacking_lr"]
DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,  n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,  n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000, n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run3_large_opt_sample"),
]

import torch
sys.path.append("../external/LUNAR")
import LUNAR
import variables as var
from optuna_utils import run_study
from sklearn.metrics import roc_auc_score

MODEL_TYPE = "LUNAR_Epochs_v2"
SAMPLE_TYPES = ["UNIFORM", "SUBSPACE", "MIXED"]
EPOCH_GRID = [25, 50, 75, 100, 150, 200, 250, 300]


def tune_lunar_base(train_x, train_y, val_x, val_y, dataset, seed, n_trials, results_dir):
    def objective(trial):
        params = {
            "k": trial.suggest_int("k", 5, 150, log=True),
            "samples": trial.suggest_categorical("samples", SAMPLE_TYPES),
            "lr": trial.suggest_float("lr", 1e-4, 1e-1, log=True),
            "wd": trial.suggest_float("wd", 1e-4, 1.0, log=True),
            "epsilon": trial.suggest_float("epsilon", 0.01, 0.5),
            "proportion": trial.suggest_int("proportion", 1, 2),
            "n_epochs": trial.suggest_int("n_epochs", 50, 300, step=25),
        }
        var.lr = params["lr"]; var.wd = params["wd"]; var.epsilon = params["epsilon"]
        var.proportion = params["proportion"]; var.n_epochs = params["n_epochs"]
        out = LUNAR.run(train_x, train_y, val_x, val_y, val_x, val_y, dataset, seed, params["k"], params["samples"], train_new_model=True)
        return roc_auc_score(val_y, out.numpy())
    return run_study(objective, f"LUNAR_base_epochs_{dataset}", seed, n_trials, results_dir=results_dir).best_params

records = []
for dataset in DATASETS:
    for run_cfg in RUN_CONFIGS:
        opt_train_x, opt_train_y, opt_val_x, opt_val_y = make_optuna_subsample(dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"])
        best_base = tune_lunar_base(opt_train_x, opt_train_y, opt_val_x, opt_val_y, dataset, SEED, N_TRIALS, RESULTS_DIR)
        train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(dataset, SEED, run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"], max_nodes_budget=50_000_000, k=best_base["k"])
        for n_epochs in EPOCH_GRID:
            var.lr = best_base["lr"]; var.wd = best_base["wd"]; var.epsilon = best_base["epsilon"]
            var.proportion = best_base["proportion"]; var.n_epochs = n_epochs
            original_device = var.device; var.device = torch.device("cpu")
            t0 = time.time(); out_val = LUNAR.run(train_x, train_y, val_x, val_y, val_x, val_y, dataset, SEED, best_base["k"], best_base["samples"], train_new_model=True); runtime_train = time.time() - t0
            out_test = LUNAR.run(train_x, train_y, val_x, val_y, test_x, test_y, dataset, SEED, best_base["k"], best_base["samples"], train_new_model=True)
            var.device = original_device
            scores_val = minmax_scale_scores(out_val.numpy())
            scores_test = minmax_scale_scores(out_test.numpy())
            thr, *_ = find_best_f1_threshold(val_y, scores_val)
            rec = build_experiment_record(dataset, DATASET_VERSION, SPLIT_METHOD, SEED, PREPROCESSING_VERSION, MODEL_TYPE, "none", {**best_base, "n_epochs": n_epochs}, thr, scores_test, test_y, runtime_train, 0.0, get_memory_mb(), f"{run_cfg['notes']} | n_epochs={n_epochs}")
            save_record_json(rec, RESULTS_DIR, run_cfg["run_index"], f"{MODEL_TYPE}_{n_epochs}epochs", dataset)
            records.append(rec)

history = pd.DataFrame(records)
history.to_csv(RESULTS_DIR / "lunar_epochs_v2_history.csv", index=False)
history["n_epochs"] = history["notes"].str.extract(r"n_epochs=(\d+)").astype(int)
for metric in ["AUC_ROC", "AUC_PR", "Precision", "Recall", "F1"]:
    fig = px.line(history, x="n_epochs", y=metric, color="dataset_name", markers=True, title=f"{metric} vs n_epochs (tuned base LUNAR)")
    fig.write_html(RESULTS_DIR / f"{metric.lower()}_vs_epochs_v2.html")
history
